# MODEL

In this file, readind the output data from feature_engineering.ipynb, we will:

1. Define the PanelSplit
2. Define the model
3. Apply cross_val_fit_predict to do the prediction 
4. Evaluation

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from panelsplit.cross_validation import PanelSplit
import matplotlib.pyplot as plt
from sklearn.metrics import (
    classification_report, f1_score, roc_auc_score, 
    roc_curve, auc, precision_recall_curve, average_precision_score
)

First we load the data:

In [2]:
df = pd.read_parquet("../data_clean/final_data.parquet")

### DEFINE THE TARGET VARIABLE AND THE FEATURES

In [3]:
def prepare_ml_experiment(df, features_to_use=None, target_col='target_2m'):
    df_temp = df.copy()
    
    # HIDE THE FUTURE: Mask last 2 months per country, because we won't have the target for those months at prediction time.
    last_2_months_mask = df_temp.groupby('iso3').cumcount(ascending=False) < 2
    df_temp[target_col] = df_temp[target_col].astype(float)
    df_temp.loc[last_2_months_mask, target_col] = np.nan

    # FEATURE SELECTION
    blacklist = [target_col, 'allocation_elegible', 'notes_acled', 'hdx_alert_level']
    features_cols = [col for col in features_to_use if col not in blacklist]

    # CLEANING: Ensure we only keep rows where both features and target exist
    df_model = df_temp[features_cols + [target_col]].dropna()
    
    X = df_model[features_cols]
    y = df_model[target_col]
    
    return X, y

Now we are going to define different groups of features and check with which combination the model performs better:

In [ ]:
# Only raw variables from IDMC and ACLED
features_baseline = ['monthly_displacement','fatalities', 'event_count', 'allocation_elegible']

# Features derived from IDMC (lags, rolling means, etc.) and from ACLED (lags, rolling means and also the "displacement score" that we created)
features_derived_humanitarian_impact = ['rolling_3m_displacements', 'disp_6m_avg', 'monthly_displacement_lag1', 'monthly_displacement_lag2' ,
                                        'fatalities_lag1', 'fat_6m_avg', 'acled_disp_score_max', 'acled_disp_score_mean', 'acled_disp_events_count', 
                                        'acled_disp_events_ratio']
# Features from EconAI data and HDX Signals
features_risk_alerts = ['risk_3', 'risk_12', 'logfat_risk_3', 'logfat_risk_12', 'hdx_value', 'risk_gt_06', 'hdx_med_high_count', 
                        'hdx_3m_sum', 'risk_3_lag1', 'risk_12_lag1', 'logfat_risk_3_lag1', 'logfat_risk_12_lag1', 'hdx_alert_max', 
                        'hdx_alert_sum', 'hdx_alert_mean', 'hdx_medium_count', 'hdx_high_count']

# Only features from INFORM Index
features_inform = ['INFORM', 'VU', 'CC', 'HA']

# Only features from Severity Index
features_severity = ['inform_severity_index']

# Only the signals that CERF has identified
features_cerf_signals = ['p_sig1', 'p_sig2', 'p_sig3', 'protracted_signal', 'h_sig1', 'h_sig2', 'hard_onset_signal', 'early_signal']

We only want to keep the most relevant features: REGARDING THE F1-SCORE

In [11]:
from sklearn.inspection import permutation_importance
from sklearn.metrics import f1_score

def get_valuable_features_f1_pure(df, feature_list, target_col='target_2m', top_n=5):
    if len(feature_list) <= 3:
        return feature_list
        
    X, y = prepare_ml_experiment(df, features_to_use=feature_list, target_col=target_col)
    
    time_axis = pd.to_datetime(X.index.get_level_values('month'))
    max_date = time_axis.max()
    cutoff_date = max_date - pd.DateOffset(years=2)
    
    X_train_safe = X[time_axis < cutoff_date]
    y_train_safe = y[time_axis < cutoff_date]
    
    # Entrenamos un modelo robusto con pesos balanceados
    model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)
    model.fit(X_train_safe, y_train_safe)
    
    # 🎯 Evaluamos el impacto directo de cada feature sobre el F1-Score matemático
    result = permutation_importance(
        model, X_train_safe, y_train_safe, 
        scoring='f1',       # <--- Forzamos que evalúe basándose en F1
        n_repeats=5, 
        random_state=42, 
        n_jobs=-1
    )
    
    # Extraemos la importancia media de la permutación
    importances = pd.Series(result.importances_mean, index=X.columns)
    best_features = importances.sort_values(ascending=False).head(top_n).index.tolist()
    
    return best_features

In [12]:
best_features_derived_humanitarian_impact = get_valuable_features_f1_pure(df, features_derived_humanitarian_impact)
best_features_risk_alerts = get_valuable_features_f1_pure(df, features_risk_alerts)
best_features_cerf_signals = get_valuable_features_f1_pure(df, features_cerf_signals)

In [13]:
print("Best features from Derived Humanitarian Impact:", best_features_derived_humanitarian_impact)
print("Best features from Risk Alerts:", best_features_risk_alerts)
print("Best features from CERF Signals:", best_features_cerf_signals)

Best features from Derived Humanitarian Impact: ['disp_6m_avg', 'fat_6m_avg', 'fatalities_lag1', 'rolling_3m_displacements', 'acled_disp_score_mean']
Best features from Risk Alerts: ['logfat_risk_3', 'logfat_risk_3_lag1', 'logfat_risk_12', 'risk_12', 'risk_3']
Best features from CERF Signals: ['early_signal', 'h_sig2', 'p_sig1', 'h_sig1', 'p_sig3']


Let's now run all the experiments and save the results:

In [7]:
from sklearn.dummy import DummyClassifier  # <-- Importamos esto para el random
import os

def run_and_save_experiment(experiment_name, file_prefix, feature_list, df, model=None, results_folder="results"):
    print("\n" + "="*60)
    print(f"RUNNING EXPERIMENT: {experiment_name}")
    print(f"Features count: {len(feature_list)}")
    print("="*60)
    
    os.makedirs(results_folder, exist_ok=True)
    
    # 1. Prepare data
    X, y = prepare_ml_experiment(df, features_to_use=feature_list)
    periods = X.index.get_level_values('month')

    # 2. Split strategy
    panel_split = PanelSplit(periods=periods, n_splits=24, test_size=1, gap=1)

    # 3. Initialize Model (AQUÍ ESTÁ EL TRUCO)
    # Si le pasas un modelo por parámetro usa ese, si no, usa tu RF de siempre
    clf = model if model is not None else RandomForestClassifier(
        n_estimators=200, max_depth=10, class_weight='balanced', random_state=42, n_jobs=-1
    )

    all_y_true, all_y_prob, all_y_pred = [], [], []
    all_iso3, all_months = [], []

    # 4. Cross-Validation Loop
    for train_idx, test_idx in panel_split.split(X):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        
        clf.fit(X_train, y_train)
        
        probs = clf.predict_proba(X_test)[:, 1]
        preds = clf.predict(X_test)
        
        all_y_true.extend(y_test.values)
        all_y_prob.extend(probs)
        all_y_pred.extend(preds)

        all_iso3.extend(X_test.index.get_level_values('iso3'))
        all_months.extend(X_test.index.get_level_values('month'))

    # 5. SAVE FILE 1: Predictions
    df_preds = pd.DataFrame({
        'iso3': all_iso3,
        'month': all_months,
        'y_true': all_y_true,
        'y_prob': all_y_prob
    })
    pred_path = os.path.join(results_folder, f"{file_prefix}_predictions.csv")
    df_preds.to_csv(pred_path, index=False)
    print(f"✅ Saved predictions to: {pred_path}")
    
    # 6. SAVE FILE 2: Feature Importance (Solo si el modelo las tiene)
    if hasattr(clf, "feature_importances_"):
        df_feat = pd.DataFrame({
            'feature': X.columns,
            'importance': clf.feature_importances_
        }).sort_values(by='importance', ascending=False)
        feat_path = os.path.join(results_folder, f"{file_prefix}_feature_importance.csv")
        df_feat.to_csv(feat_path, index=False)
        print(f"✅ Saved feature importance to: {feat_path}")
    else:
        print(f"⚠️ Skipped feature importance (Este modelo no utiliza features)")
    
    # 7. Quick report
    print(f"📊 F1: {f1_score(all_y_true, all_y_pred):.3f} | AUC: {roc_auc_score(all_y_true, all_y_prob):.3f}")

Our first experiment is going to be our benchmark model. This id going to be a random classifier respecting the proportions of the clases.

In [8]:
run_and_save_experiment(
    experiment_name="Random Benchmark (Stratified)", 
    file_prefix="benchmark_random", 
    feature_list=[], 
    df=df,
    model=DummyClassifier(strategy='stratified', random_state=42) 
)


RUNNING EXPERIMENT: Random Benchmark (Stratified)
Features count: 0
✅ Saved predictions to: results/benchmark_random_predictions.csv
⚠️ Skipped feature importance (Este modelo no utiliza features)
📊 F1: 0.000 | AUC: 0.499


Now let's try training the model with all the possible combination of features

In [ ]:
import itertools
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

# =====================================================================
# DEFINE DISCRETE OPTION STATES FOR EACH FEATURE BLOCK
# =====================================================================
# Each block can either be skipped (None), use its General array, 
# or use its optimized BEST array. Mutual exclusivity is enforced here.
feature_dimensions = {
    "Baseline": [
        None, 
        {"label": "Baseline", "suffix": "only_baseline", "features": features_baseline}
    ],
    "Derived Impact": [
        None, 
        {"label": "Derived Humanitarian Impact", "suffix": "derived_humanitarian_impact", "features": features_derived_humanitarian_impact},
        {"label": "Derived Humanitarian Impact BEST", "suffix": "derived_humanitarian_impact_best", "features": best_features_derived_humanitarian_impact}
    ],
    "Risk Alerts": [
        None, 
        {"label": "Only Risk Alerts", "suffix": "only_risk_alerts", "features": features_risk_alerts},
        {"label": "Only Risk Alerts BEST", "suffix": "only_risk_alerts_best", "features": best_features_risk_alerts}
    ],
    "INFORM": [
        None, 
        {"label": "Only INFORM", "suffix": "only_inform", "features": features_inform}
    ],
    "Severity": [
        None, 
        {"label": "Only Severity", "suffix": "only_severity", "features": features_severity}
    ],
    "Signals (CERF)": [
        None, 
        {"label": "Only Signals (CERF)", "suffix": "only_signals_cerf", "features": features_cerf_signals},
        {"label": "Only Signals (CERF) BEST", "suffix": "only_signals_cerf_best", "features": best_features_cerf_signals}
    ]
}

# =====================================================================
# COMPUTE THE CARTESIAN PRODUCT & BUILD CONFIGURATIONS (FIXED)
# =====================================================================
experiment_configs = []

keys = list(feature_dimensions.keys())
all_dimension_options = [feature_dimensions[k] for k in keys]

for combo in itertools.product(*all_dimension_options):
    # Filter out inactive dimensions (None values)
    active_blocks = [block for block in combo if block is not None]
    
    # Skip completely empty sets
    if not active_blocks:
        continue
        
    # Extract structural labels and filename chunks
    block_names = [b["label"] for b in active_blocks]
    block_suffixes = [b["suffix"] for b in active_blocks]
    
    # Format specific edge-case names if an experiment combines all 6 groups
    if len(active_blocks) == 6:
        # PURE GENERAL: No block contains the substring "BEST"
        is_pure_general = not any("BEST" in b["label"] for b in active_blocks)
        
        # ALL BEST: If a block *can* be optimized (Derived, Risk, Signals), it *is* optimized.
        # This means no unoptimized versions of those 3 blocks are present.
        has_no_raw_derived = "Derived Humanitarian Impact" not in block_names
        has_no_raw_risk = "Only Risk Alerts" not in block_names
        has_no_raw_signals = "Only Signals (CERF)" not in block_names
        is_all_best = has_no_raw_derived and has_no_raw_risk and has_no_raw_signals
        
        if is_all_best:
            name = "ALL FEATURES (With BEST Options)"
            prefix = "all_features_best"
        elif is_pure_general:
            name = "ALL FEATURES (Baseline + Derived Impact + Risk Alerts + INFORM + Severity + Signals CERF)"
            prefix = "all_features"
        else:
            name = f"ALL FEATURES (Hybrid Mix: {' + '.join(block_names)})"
            prefix = f"all_features_hybrid_{'_'.join(block_suffixes)}"
    else:
        # Standard dynamic parsing for combinations of 1 to 5 feature groups
        name = " + ".join(block_names)
        prefix = "_".join(block_suffixes)
        
    # Safely aggregate unique features across the selected active blocks
    combined_features = []
    for b in active_blocks:
        for feature_name in b["features"]:
            if feature_name not in combined_features:
                combined_features.append(feature_name)
                
    experiment_configs.append({
        "name": name,
        "prefix": prefix,
        "features": combined_features
    })

# =====================================================================
# EXECUTE THE AUTOMATED SEAMLESS TRAINING GRID LOOP
# =====================================================================
print(f"🚀 Successfully generated {len(experiment_configs)} valid experimental configurations.")
print("CRITICAL GUARDRAIL: No experiment will mix a general feature block with its own BEST variant.\n")

for config in experiment_configs:
    run_and_save_experiment(
        experiment_name=config["name"],
        file_prefix=config["prefix"],
        feature_list=config["features"],
        df=df 
    )

🚀 Successfully generated 215 valid experimental configurations.
CRITICAL GUARDRAIL: No experiment will mix a general feature block with its own BEST variant.


RUNNING EXPERIMENT: Only Signals (CERF)
Features count: 8
✅ Saved predictions to: results/only_signals_cerf_predictions.csv
✅ Saved feature importance to: results/only_signals_cerf_feature_importance.csv
📊 F1: 0.154 | AUC: 0.821

RUNNING EXPERIMENT: Only Signals (CERF) BEST
Features count: 5
✅ Saved predictions to: results/only_signals_cerf_best_predictions.csv
✅ Saved feature importance to: results/only_signals_cerf_best_feature_importance.csv
📊 F1: 0.143 | AUC: 0.582

RUNNING EXPERIMENT: Only Severity
Features count: 1
✅ Saved predictions to: results/only_severity_predictions.csv
✅ Saved feature importance to: results/only_severity_feature_importance.csv
📊 F1: 0.119 | AUC: 0.629

RUNNING EXPERIMENT: Only Severity + Only Signals (CERF)
Features count: 9
✅ Saved predictions to: results/only_severity_only_signals_cerf_predictions

In [9]:
experiment_configs = [
    # ========================================================
    # --- SOLO GROUPS (6) ---
    # ========================================================
    {
        "name": "Baseline", 
        "prefix": "only_baseline", 
        "features": features_baseline
    },
    {
        "name": "Derived Humanitarian Impact", 
        "prefix": "only_derived_humanitarian_impact", 
        "features": features_derived_humanitarian_impact
    },
    {
        "name": "Only Risk Alerts", 
        "prefix": "only_risk_alerts", 
        "features": features_risk_alerts
    },
    {
        "name": "Only INFORM", 
        "prefix": "only_inform", 
        "features": features_inform
    },
    {
        "name": "Only Severity", 
        "prefix": "only_severity", 
        "features": features_severity
    },
    {
        "name": "Only Signals (CERF)", 
        "prefix": "only_signals_cerf", 
        "features": features_cerf_signals
    },

    # ========================================================
    # --- COMBINED GROUPS OF 2 (15) ---
    # ========================================================
    {
        "name": "Baseline + Derived Humanitarian Impact", 
        "prefix": "baseline_derived_humanitarian_impact", 
        "features": features_baseline + features_derived_humanitarian_impact
    },
    {
        "name": "Baseline + Risk Alerts", 
        "prefix": "baseline_risk_alerts", 
        "features": features_baseline + features_risk_alerts
    },
    {
        "name": "Baseline + INFORM", 
        "prefix": "baseline_inform", 
        "features": features_baseline + features_inform
    },
    {
        "name": "Baseline + Severity", 
        "prefix": "baseline_severity", 
        "features": features_baseline + features_severity
    },
    {
        "name": "Baseline + Signals (CERF)", 
        "prefix": "baseline_signals_cerf", 
        "features": features_baseline + features_cerf_signals
    },
    {
        "name": "Derived Humanitarian Impact + Risk Alerts", 
        "prefix": "derived_humanitarian_impact_risk_alerts", 
        "features": features_derived_humanitarian_impact + features_risk_alerts
    },
    {
        "name": "Derived Humanitarian Impact + INFORM", 
        "prefix": "derived_humanitarian_impact_inform", 
        "features": features_derived_humanitarian_impact + features_inform
    },
    {
        "name": "Derived Humanitarian Impact + Severity", 
        "prefix": "derived_humanitarian_impact_severity", 
        "features": features_derived_humanitarian_impact + features_severity
    },
    {
        "name": "Derived Humanitarian Impact + Signals (CERF)", 
        "prefix": "derived_humanitarian_impact_signals_cerf", 
        "features": features_derived_humanitarian_impact + features_cerf_signals
    },
    {
        "name": "Risk Alerts + INFORM", 
        "prefix": "risk_alerts_inform", 
        "features": features_risk_alerts + features_inform
    },
    {
        "name": "Risk Alerts + Severity", 
        "prefix": "risk_alerts_severity", 
        "features": features_risk_alerts + features_severity
    },
    {
        "name": "Risk Alerts + Signals (CERF)", 
        "prefix": "risk_alerts_signals_cerf", 
        "features": features_risk_alerts + features_cerf_signals
    },
    {
        "name": "INFORM + Severity", 
        "prefix": "inform_severity", 
        "features": features_inform + features_severity
    },
    {
        "name": "INFORM + Signals (CERF)", 
        "prefix": "inform_signals_cerf", 
        "features": features_inform + features_cerf_signals
    },
    {
        "name": "Severity + Signals (CERF)", 
        "prefix": "severity_signals_cerf", 
        "features": features_severity + features_cerf_signals
    },

    # ========================================================
    # --- COMBINED GROUPS OF 3 (20) ---
    # ========================================================
    {
        "name": "Baseline + Derived Impact + Risk Alerts", 
        "prefix": "baseline_derived_impact_risk_alerts", 
        "features": features_baseline + features_derived_humanitarian_impact + features_risk_alerts
    },
    {
        "name": "Baseline + Derived Impact + INFORM", 
        "prefix": "baseline_derived_impact_inform", 
        "features": features_baseline + features_derived_humanitarian_impact + features_inform
    },
    {
        "name": "Baseline + Derived Impact + Severity", 
        "prefix": "baseline_derived_impact_severity", 
        "features": features_baseline + features_derived_humanitarian_impact + features_severity
    },
    {
        "name": "Baseline + Derived Impact + Signals (CERF)", 
        "prefix": "baseline_derived_impact_signals_cerf", 
        "features": features_baseline + features_derived_humanitarian_impact + features_cerf_signals
    },
    {
        "name": "Baseline + Risk Alerts + INFORM", 
        "prefix": "baseline_risk_alerts_inform", 
        "features": features_baseline + features_risk_alerts + features_inform
    },
    {
        "name": "Baseline + Risk Alerts + Severity", 
        "prefix": "baseline_risk_alerts_severity", 
        "features": features_baseline + features_risk_alerts + features_severity
    },
    {
        "name": "Baseline + Risk Alerts + Signals (CERF)", 
        "prefix": "baseline_risk_alerts_signals_cerf", 
        "features": features_baseline + features_risk_alerts + features_cerf_signals
    },
    {
        "name": "Baseline + INFORM + Severity", 
        "prefix": "baseline_inform_severity", 
        "features": features_baseline + features_inform + features_severity
    },
    {
        "name": "Baseline + INFORM + Signals (CERF)", 
        "prefix": "baseline_inform_signals_cerf", 
        "features": features_baseline + features_inform + features_cerf_signals
    },
    {
        "name": "Baseline + Severity + Signals (CERF)", 
        "prefix": "baseline_severity_signals_cerf", 
        "features": features_baseline + features_severity + features_cerf_signals
    },
    {
        "name": "Derived Impact + Risk Alerts + INFORM", 
        "prefix": "derived_impact_risk_alerts_inform", 
        "features": features_derived_humanitarian_impact + features_risk_alerts + features_inform
    },
    {
        "name": "Derived Impact + Risk Alerts + Severity", 
        "prefix": "derived_impact_risk_alerts_severity", 
        "features": features_derived_humanitarian_impact + features_risk_alerts + features_severity
    },
    {
        "name": "Derived Impact + Risk Alerts + Signals (CERF)", 
        "prefix": "derived_impact_risk_alerts_signals_cerf", 
        "features": features_derived_humanitarian_impact + features_risk_alerts + features_cerf_signals
    },
    {
        "name": "Derived Impact + INFORM + Severity", 
        "prefix": "derived_impact_inform_severity", 
        "features": features_derived_humanitarian_impact + features_inform + features_severity
    },
    {
        "name": "Derived Impact + INFORM + Signals (CERF)", 
        "prefix": "derived_impact_inform_signals_cerf", 
        "features": features_derived_humanitarian_impact + features_inform + features_cerf_signals
    },
    {
        "name": "Derived Impact + Severity + Signals (CERF)", 
        "prefix": "derived_impact_severity_signals_cerf", 
        "features": features_derived_humanitarian_impact + features_severity + features_cerf_signals
    },
    {
        "name": "Risk Alerts + INFORM + Severity", 
        "prefix": "risk_alerts_inform_severity", 
        "features": features_risk_alerts + features_inform + features_severity
    },
    {
        "name": "Risk Alerts + INFORM + Signals (CERF)", 
        "prefix": "risk_alerts_inform_signals_cerf", 
        "features": features_risk_alerts + features_inform + features_cerf_signals
    },
    {
        "name": "Risk Alerts + Severity + Signals (CERF)", 
        "prefix": "risk_alerts_severity_signals_cerf", 
        "features": features_risk_alerts + features_severity + features_cerf_signals
    },
    {
        "name": "INFORM + Severity + Signals (CERF)", 
        "prefix": "inform_severity_signals_cerf", 
        "features": features_inform + features_severity + features_cerf_signals
    },

    # ========================================================
    # --- COMBINED GROUPS OF 4 (15) ---
    # ========================================================
    {
        "name": "Baseline + Derived Impact + Risk Alerts + INFORM", 
        "prefix": "baseline_derived_impact_risk_alerts_inform", 
        "features": features_baseline + features_derived_humanitarian_impact + features_risk_alerts + features_inform
    },
    {
        "name": "Baseline + Derived Impact + Risk Alerts + Severity", 
        "prefix": "baseline_derived_impact_risk_alerts_severity", 
        "features": features_baseline + features_derived_humanitarian_impact + features_risk_alerts + features_severity
    },
    {
        "name": "Baseline + Derived Impact + Risk Alerts + Signals (CERF)", 
        "prefix": "baseline_derived_impact_risk_alerts_signals_cerf", 
        "features": features_baseline + features_derived_humanitarian_impact + features_risk_alerts + features_cerf_signals
    },
    {
        "name": "Baseline + Derived Impact + INFORM + Severity", 
        "prefix": "baseline_derived_impact_inform_severity", 
        "features": features_baseline + features_derived_humanitarian_impact + features_inform + features_severity
    },
    {
        "name": "Baseline + Derived Impact + INFORM + Signals (CERF)", 
        "prefix": "baseline_derived_impact_inform_signals_cerf", 
        "features": features_baseline + features_derived_humanitarian_impact + features_inform + features_cerf_signals
    },
    {
        "name": "Baseline + Derived Impact + Severity + Signals (CERF)", 
        "prefix": "baseline_derived_impact_severity_signals_cerf", 
        "features": features_baseline + features_derived_humanitarian_impact + features_severity + features_cerf_signals
    },
    {
        "name": "Baseline + Risk Alerts + INFORM + Severity", 
        "prefix": "baseline_risk_alerts_inform_severity", 
        "features": features_baseline + features_risk_alerts + features_inform + features_severity
    },
    {
        "name": "Baseline + Risk Alerts + INFORM + Signals (CERF)", 
        "prefix": "baseline_risk_alerts_inform_signals_cerf", 
        "features": features_baseline + features_risk_alerts + features_inform + features_cerf_signals
    },
    {
        "name": "Baseline + Risk Alerts + Severity + Signals (CERF)", 
        "prefix": "baseline_risk_alerts_severity_signals_cerf", 
        "features": features_baseline + features_risk_alerts + features_severity + features_cerf_signals
    },
    {
        "name": "Baseline + INFORM + Severity + Signals (CERF)", 
        "prefix": "baseline_inform_severity_signals_cerf", 
        "features": features_baseline + features_inform + features_severity + features_cerf_signals
    },
    {
        "name": "Derived Impact + Risk Alerts + INFORM + Severity", 
        "prefix": "derived_impact_risk_alerts_inform_severity", 
        "features": features_derived_humanitarian_impact + features_risk_alerts + features_inform + features_severity
    },
    {
        "name": "Derived Impact + Risk Alerts + INFORM + Signals (CERF)", 
        "prefix": "derived_impact_risk_alerts_inform_signals_cerf", 
        "features": features_derived_humanitarian_impact + features_risk_alerts + features_inform + features_cerf_signals
    },
    {
        "name": "Derived Impact + Risk Alerts + Severity + Signals (CERF)", 
        "prefix": "derived_impact_risk_alerts_severity_signals_cerf", 
        "features": features_derived_humanitarian_impact + features_risk_alerts + features_severity + features_cerf_signals
    },
    {
        "name": "Derived Impact + INFORM + Severity + Signals (CERF)", 
        "prefix": "derived_impact_inform_severity_signals_cerf", 
        "features": features_derived_humanitarian_impact + features_inform + features_severity + features_cerf_signals
    },
    {
        "name": "Risk Alerts + INFORM + Severity + Signals (CERF)", 
        "prefix": "risk_alerts_inform_severity_signals_cerf", 
        "features": features_risk_alerts + features_inform + features_severity + features_cerf_signals
    },

    # ========================================================
    # --- COMBINED GROUPS OF 5 (6) ---
    # ========================================================
    {
        "name": "Baseline + Derived Impact + Risk Alerts + INFORM + Severity", 
        "prefix": "baseline_derived_impact_risk_alerts_inform_severity", 
        "features": features_baseline + features_derived_humanitarian_impact + features_risk_alerts + features_inform + features_severity
    },
    {
        "name": "Baseline + Derived Impact + Risk Alerts + INFORM + Signals (CERF)", 
        "prefix": "baseline_derived_impact_risk_alerts_inform_signals_cerf", 
        "features": features_baseline + features_derived_humanitarian_impact + features_risk_alerts + features_inform + features_cerf_signals
    },
    {
        "name": "Baseline + Derived Impact + Risk Alerts + Severity + Signals (CERF)", 
        "prefix": "baseline_derived_impact_risk_alerts_severity_signals_cerf", 
        "features": features_baseline + features_derived_humanitarian_impact + features_risk_alerts + features_severity + features_cerf_signals
    },
    {
        "name": "Baseline + Derived Impact + INFORM + Severity + Signals (CERF)", 
        "prefix": "baseline_derived_impact_inform_severity_signals_cerf", 
        "features": features_baseline + features_derived_humanitarian_impact + features_inform + features_severity + features_cerf_signals
    },
    {
        "name": "Baseline + Risk Alerts + INFORM + Severity + Signals (CERF)", 
        "prefix": "baseline_risk_alerts_inform_severity_signals_cerf", 
        "features": features_baseline + features_risk_alerts + features_inform + features_severity + features_cerf_signals
    },
    {
        "name": "Derived Impact + Risk Alerts + INFORM + Severity + Signals (CERF)", 
        "prefix": "derived_impact_risk_alerts_inform_severity_signals_cerf", 
        "features": features_derived_humanitarian_impact + features_risk_alerts + features_inform + features_severity + features_cerf_signals
    },

    # ========================================================
    # --- ALL FEATURES (1) ---
    # ========================================================
    {
        "name": "ALL FEATURES (Baseline + Derived Impact + Risk Alerts + INFORM + Severity + Signals CERF)", 
        "prefix": "all_features", 
        "features": features_baseline + features_derived_humanitarian_impact + features_risk_alerts + features_inform + features_severity + features_cerf_signals
    },
]

# Launch the automatic training loop
for config in experiment_configs:
    run_and_save_experiment(
        experiment_name=config["name"],
        file_prefix=config["prefix"],
        feature_list=config["features"],
        df=df 
    )


RUNNING EXPERIMENT: Baseline
Features count: 3
✅ Saved predictions to: results/only_baseline_predictions.csv
✅ Saved feature importance to: results/only_baseline_feature_importance.csv
📊 F1: 0.234 | AUC: 0.811

RUNNING EXPERIMENT: Derived Humanitarian Impact
Features count: 10
✅ Saved predictions to: results/only_derived_humanitarian_impact_predictions.csv
✅ Saved feature importance to: results/only_derived_humanitarian_impact_feature_importance.csv
📊 F1: 0.272 | AUC: 0.893

RUNNING EXPERIMENT: Only Risk Alerts
Features count: 17
✅ Saved predictions to: results/only_risk_alerts_predictions.csv
✅ Saved feature importance to: results/only_risk_alerts_feature_importance.csv
📊 F1: 0.191 | AUC: 0.887

RUNNING EXPERIMENT: Only INFORM
Features count: 4
✅ Saved predictions to: results/only_inform_predictions.csv
✅ Saved feature importance to: results/only_inform_feature_importance.csv
📊 F1: 0.229 | AUC: 0.901

RUNNING EXPERIMENT: Only Severity
Features count: 1
✅ Saved predictions to: results